# Lab session 2

1. **networkx primer**: turn a causality matrix into a directed graph.
2. **Comments on Pairwise Granger loop**
3. **`TimeSeriesSplit` + comments on `GridSearchCV`**: for tuning parameters of models.

## 1. Networkx primer

4-asset adjacency matrix. Entry `A.loc[i, j] = l > 0` means *i Granger-causes j at lag l*.

For HW1 replace `A` with your own 10×10 matrix from the causality loop below.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

A = pd.DataFrame(
    [[0, 2, 0, 0],
     [0, 0, 1, 3],
     [0, 0, 0, 0],
     [1, 0, 2, 0]],
    index=['A', 'B', 'C', 'D'], columns=['A', 'B', 'C', 'D'])
A

In [ ]:
G = nx.from_pandas_adjacency(A, create_using=nx.DiGraph)
for u, v, d in G.edges(data=True):
    d['lag'] = int(A.loc[u, v])

pos = nx.circular_layout(G)
fig, ax = plt.subplots(figsize=(6, 6))
nx.draw_networkx_nodes(G, pos, node_color='lightblue', node_size=1200, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=14, ax=ax)
nx.draw_networkx_edges(G, pos, arrows=True, arrowsize=22,
                       connectionstyle='arc3,rad=0.12', ax=ax)
edge_labels = {(u, v): d['lag'] for u, v, d in G.edges(data=True)}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, ax=ax)
ax.set_axis_off()
plt.show()

In [ ]:
# Some suggested Network metrics:
# out_degree(), in_degree(), pagerank(). Check NetworkX documentation
#   'Who drives the price?'  -> highest out-degree (and optionally unweighted pagerank)
#   'Where does fear spill?' -> highest out-degree in the volatility network
# Lag is not causal strength, so ignore edge weights for PageRank here.


## 2. Pairwise Granger loop

Build your causality tables,  each for {weekly returns, monthly returns, weekly EWMA-vol, monthly EWMA-vol} using grangercausalitytests() for pairs of potential (cause,effect) time series.


## 3. `TimeSeriesSplit` + `GridSearchCV`

For tuning hyperparameters (hidden layers size, depth, activation,...) must further split training set into Train and Validation. Then test in a held-out set. Important:

- `TimeSeriesSplit` (never `KFold` on time series).
- Tune on validation folds, then evaluate **once** on a held-out test set.

We demo on synthetic regression data; for the HW swap in features and target.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

rng = np.random.default_rng(seed=42)

# Synthetic features + target
n = 500
X = rng.normal(size=(n, 4))
y_signal = (np.sin(X[:, 0]) + 0.5 * X[:, 1] - 0.3 * X[:, 2]**2
            + 0.2 * rng.normal(size=n))

# Held-out test = last 20% (chronological)
split = int(0.8 * n)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y_signal[:split], y_signal[split:]

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

# Visualize what the splits look like
fig, ax = plt.subplots(figsize=(8, 2.5))
for i, (tr, va) in enumerate(tscv.split(X_train)):
    ax.scatter(tr, [i] * len(tr), c='steelblue', s=8, label='train' if i == 0 else None)
    ax.scatter(va, [i] * len(va), c='orange',   s=8, label='val'   if i == 0 else None)
ax.set_yticks(range(5)); ax.set_xlabel('sample index'); ax.set_ylabel('fold')
ax.legend(loc='upper left'); ax.set_title('TimeSeriesSplit: train always precedes val')
plt.show()

Next the GridSearch for MLPRegressor has the form

search = GridSearchCV(
    MLPRegressor(max_iter=2000, shuffle=False, random_state=0),
    grid, cv=tscv, scoring='neg_mean_squared_error', n_jobs=1,
)

where grid is your defined parameters and range of values.
Note that shuffle=False


**For LSTM/GP:** `GridSearchCV` does not wrap a Keras LSTM cleanly anymore (write a manual loop over `(timesteps, units, layers)` using the same `TimeSeriesSplit` folds).

For GP, grid-search the kernel choice and `alpha`; the kernel hyperparameters are optimized inside `fit()` via marginal likelihood.